In [25]:
import pandas as pd

In [26]:
df = pd.read_csv('job.csv')

In [27]:
df.head()

,Unnamed: 0,Resume,Category
0,0,"TensorFlow, NLP, Pytorch B.Sc 10",AI Researcher
1,1,"Deep Learning, Machine Learning, Python, SQL M...",Data Scientist
2,2,"Ethical Hacking, Cybersecurity, Linux MBA 1 De...",Cybersecurity Analyst
3,3,"Python, Pytorch, TensorFlow B.Tech 7 AWS Certi...",AI Researcher
4,4,"SQL, React, Java PhD 4",Software Engineer


In [28]:
df.shape

(17442, 3)

In [29]:
df = df.drop(['Unnamed: 0'],axis=1)

In [30]:
df.head()

,Resume,Category
0,"TensorFlow, NLP, Pytorch B.Sc 10",AI Researcher
1,"Deep Learning, Machine Learning, Python, SQL M...",Data Scientist
2,"Ethical Hacking, Cybersecurity, Linux MBA 1 De...",Cybersecurity Analyst
3,"Python, Pytorch, TensorFlow B.Tech 7 AWS Certi...",AI Researcher
4,"SQL, React, Java PhD 4",Software Engineer


In [31]:
df = df.drop_duplicates()

In [32]:
df.shape

(16093, 2)

In [33]:
df.isnull().sum()

Resume      0
Category    0
dtype: int64

In [34]:
df.head()

,Resume,Category
0,"TensorFlow, NLP, Pytorch B.Sc 10",AI Researcher
1,"Deep Learning, Machine Learning, Python, SQL M...",Data Scientist
2,"Ethical Hacking, Cybersecurity, Linux MBA 1 De...",Cybersecurity Analyst
3,"Python, Pytorch, TensorFlow B.Tech 7 AWS Certi...",AI Researcher
4,"SQL, React, Java PhD 4",Software Engineer


In [35]:
df['Category'] = df['Category'].str.strip().str.lower()

In [36]:
df['Category'].unique()

array(['ai researcher', 'data scientist', 'cybersecurity analyst',
       'software engineer', 'hr', 'designer', 'information-technology',
       'teacher', 'advocate', 'business-development', 'healthcare',
       'fitness', 'agriculture', 'bpo', 'sales', 'consultant',
       'digital-media', 'automobile', 'chef', 'finance', 'apparel',
       'engineering', 'accountant', 'construction', 'public-relations',
       'banking', 'arts', 'aviation', 'architecture', 'blockchain',
       'building and construction', 'business analyst', 'civil engineer',
       'data science', 'database', 'designing', 'devops', 'digital media',
       'dotnet developer', 'education', 'electrical engineering',
       'etl developer', 'food and beverages', 'health and fitness',
       'human resources', 'information technology', 'java developer',
       'management', 'mechanical engineer', 'network security engineer',
       'operations manager', 'pmo', 'public relations',
       'python developer', 'react develo

In [37]:
category_mapping = {
    'hr': 'human resources',
    'information technology': 'information technology',
    'data scientist': 'data science',
    'data science': 'data science',
    'accountant': 'accountant',
    'advocate': 'advocate',
    'agriculture': 'agriculture',
    'apparel': 'apparel',
    'arts': 'arts',
    'automobile': 'automobile',
    'aviation': 'aviation',
    'banking': 'banking',
    'consultant': 'consultant',
    'finance': 'finance',
    'sales': 'sales',
    'public relations': 'public relations',
    'digital media': 'digital media',
    'health and fitness': 'health and fitness',
    'healthcare': 'healthcare',
    'fitness': 'fitness',
}
df['Category'] = df['Category'].replace(category_mapping)

In [38]:
df['Category'].nunique()

66

In [39]:
df['Category'].value_counts()

Category
data science                 561
sales                        467
consultant                   460
accountant                   455
finance                      450
                            ... 
backend developer             19
machine learning engineer     18
automation testing             7
devops engineer                7
hadoop                         7
Name: count, Length: 66, dtype: int64

In [40]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

df['Resume'] = df['Resume'].apply(clean_text)

In [41]:
X = df['Resume']
y = df['Category']

In [42]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [43]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    lowercase=True,
    stop_words='english',
    max_features=10000
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [44]:
X_train_tfidf.shape

(12874, 10000)

In [45]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, recall_score, classification_report

lr = LogisticRegression(
    class_weight='balanced',
    C=0.02,
    l1_ratio=0,
    solver='lbfgs')
lr.fit(X_train_tfidf, y_train)

y_pred = lr.predict(X_test_tfidf)
y_pred_train = lr.predict(X_train_tfidf)


print('train')
print(accuracy_score(y_train, y_pred_train))
print(f1_score(y_train, y_pred_train, average='weighted'))

print('test')
print(accuracy_score(y_test, y_pred))
print(f1_score(y_test, y_pred, average='weighted'))


train
0.6558179276060276
0.6406520668076983
test
0.6169617893755824
0.6034273853887454


In [46]:
from sklearn.ensemble import RandomForestClassifier


rf = RandomForestClassifier(n_estimators=101, max_depth=4)
rf.fit(X_train_tfidf, y_train)

y_pred = rf.predict(X_test_tfidf)
y_pred_train = rf.predict(X_train_tfidf)


print('train')
print(accuracy_score(y_train, y_pred_train))
print(f1_score(y_train, y_pred_train, average='weighted'))

print('test')
print(accuracy_score(y_test, y_pred))
print(f1_score(y_test, y_pred, average='weighted'))


train
0.559422091036197
0.5189979327605335
test
0.5318421870146008
0.4882887750148444


In [48]:
import joblib

joblib.dump(
    tfidf,
    "models/tfidf.pkl"
)
joblib.dump(lr, "models/Logistic.pkl")

['models/Logistic.pkl']